In [1]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your Hugging Face token:  ········


Hugging Face token set successfully!


In [2]:
# Step 2: Load FLARE-CD test set and preprocess
from datasets import load_dataset, Dataset

# FLARE-CD dataset (Token-level Causal Annotation)
print("加载 FLARE-CD 数据集...")
ds_raw = load_dataset("ChanceFocus/flare-cd", split="test")
print(f"成功加载! 样本数: {len(ds_raw)}, 列: {ds_raw.column_names}")

# 定义标签类型
LABELS = ["O", "B-CAUSE", "I-CAUSE", "B-EFFECT", "I-EFFECT"]

def _map_row(x):
    tokens = x.get("token", [])
    labels = x.get("label", [])
    text = x.get("text", "")
    
    # 如果没有预分的 tokens，从 text 分词
    if not tokens and text:
        tokens = text.split()
    
    # 确保 tokens 和 labels 长度匹配
    if len(tokens) != len(labels):
        min_len = min(len(tokens), len(labels))
        tokens = tokens[:min_len]
        labels = labels[:min_len]
    
    return {
        "tokens": tokens,
        "labels": labels,
        "text": " ".join(tokens) if tokens else text,
        "label_types": LABELS
    }

ds = Dataset.from_list([_map_row(r) for r in ds_raw])
bad = [i for i, r in enumerate(ds) if not r["tokens"]]
print(f"空样本数: {len(bad)}")
assert len(bad) == 0, "发现空样本。"

print(f"\n第一个样本示例:")
print(f"  Tokens: {ds[0]['tokens'][:10]}...")
print(f"  Labels: {ds[0]['labels'][:10]}...")

加载 FLARE-CD 数据集...
成功加载! 样本数: 226, 列: ['id', 'query', 'answer', 'text', 'label', 'token']
空样本数: 0

第一个样本示例:
  Tokens: ['Around', '21,000', 'employees', ',', '9,000', 'of', 'whom', 'are', 'employed', 'in']...
  Labels: ['B-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT']...


In [3]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform, re
from importlib.metadata import version, PackageNotFoundError
import requests

# o3-mini配置
MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 文件命名
dataset_name = "flare_cd"
run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-cd",
    "split": "test",
    "label_types": LABELS,
    "model": MODEL,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini evaluation on FLARE-CD (Token-level Causal Annotation)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your OpenAI API key:  ········


Meta saved -> /content/flare_cd_o3_mini_metadata.json
MODEL: o3-mini | BASE_URL: https://api.openai.com/v1
OPENAI_API_KEY is set: True


In [4]:
# Step 4: Inference & evaluation loop for token-level causal annotation
import json, os, re, time
import pandas as pd
from tqdm import tqdm
import requests

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_cd_prompt(tokens):
    text = " ".join(tokens)
    return (
        "Task: Perform token-level causal annotation on the following text.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        "1. For each token, label it as:\n"
        "   - B-CAUSE: beginning of a cause phrase\n"
        "   - I-CAUSE: inside a cause phrase\n"
        "   - B-EFFECT: beginning of an effect phrase\n"
        "   - I-EFFECT: inside an effect phrase\n"
        "   - O: no causal role\n"
        "2. Return the labels as a JSON array in the same order as tokens\n"
        "3. Return ONLY the JSON array, no additional text\n\n"
        "Example output format:\n"
        '["B-CAUSE", "I-CAUSE", "O", "B-EFFECT", "I-EFFECT"]'
    )

def parse_token_labels_response(response_text: str, num_tokens: int):
    response_text = response_text.strip()
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if match:
        try:
            labels = json.loads(match.group())
            if isinstance(labels, list):
                if len(labels) > num_tokens:
                    labels = labels[:num_tokens]
                elif len(labels) < num_tokens:
                    labels = labels + ["O"] * (num_tokens - len(labels))
                return labels, True, None
        except json.JSONDecodeError:
            pass
    return None, False, "Failed to parse JSON array"

def ask_o3_mini_once(tokens):
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_cd_prompt(tokens)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a linguistic annotation expert. Respond only with valid JSON arrays."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        
        if response.status_code != 200:
            return None, False, f"API Error {response.status_code}"
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return None, False, "No choices in response"
            
        txt = result['choices'][0]['message']['content']
        txt = _strip_code_fences(txt)
        pred_labels, success, error = parse_token_labels_response(txt, len(tokens))
        return pred_labels, success, error
        
    except Exception as e:
        return None, False, str(e)

def ask_o3_mini(tokens):
    delay = 2.0
    for attempt in range(6):
        try:
            pred_labels, success, error = ask_o3_mini_once(tokens)
            if success:
                return pred_labels, True, None
            else:
                if attempt < 5:
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                return ["O"] * len(tokens), False, error
        except Exception as e:
            if attempt == 5:
                return ["O"] * len(tokens), False, str(e)
            time.sleep(delay)
            delay = min(delay * 2, 60)
    return ["O"] * len(tokens), False, "Exhausted retries"

run_tag = f"flare_cd_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path = f"{save_dir}/{run_tag}_errors.csv"

rows_done = []
done_idx = set()
if os.path.exists(pred_path):
    old = pd.read_csv(pred_path)
    if "row_idx" in old.columns:
        rows_done = old.to_dict("records")
        done_idx = set(old["row_idx"].tolist())
        print(f"[resume] loaded {len(done_idx)} completed rows.")

err_rows = []
buf = []
save_every = 20

total = len(ds)
print(f"Starting o3-mini token-level causal annotation on {total} samples...")

for i in tqdm(range(total)):
    if i in done_idx:
        continue
    x = ds[i]
    tokens = x["tokens"]
    gold_labels = x["labels"]

    try:
        pred_labels, success, error_msg = ask_o3_mini(tokens)
        if not success:
            raise RuntimeError(error_msg)
        raw = json.dumps(pred_labels)
    except Exception as e:
        pred_labels = ["O"] * len(tokens)
        raw = f"ERROR: {type(e).__name__}: {e}"
        err_rows.append({"row_idx": i, "error": raw, "text": " ".join(tokens[:50])})
        success = False

    buf.append({
        "row_idx": i,
        "text": " ".join(tokens[:50]) + ("..." if len(tokens) > 50 else ""),
        "token_count": len(tokens),
        "pred_raw": raw,
        "pred_labels": json.dumps(pred_labels),
        "gold_labels": json.dumps(gold_labels),
        "success": success
    })

    if len(buf) % save_every == 0:
        out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
        out.to_csv(pred_path, index=False)
        if err_rows:
            pd.DataFrame(err_rows).to_csv(err_path, index=False)
        print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
out.to_csv(pred_path, index=False)
if err_rows:
    pd.DataFrame(err_rows).to_csv(err_path, index=False)
print(f"[done] o3-mini CD evaluation completed -> {pred_path}")
if os.path.exists(err_path):
    err_count = len(pd.read_csv(err_path)) if os.path.getsize(err_path) > 0 else 0
    print(f"[errors] {err_count} errors logged -> {err_path}")

Starting o3-mini token-level causal annotation on 226 samples...


  9%|██████▉                                                                        | 20/226 [06:23<1:03:09, 18.40s/it]

[checkpoint] saved 20/226 -> /content/flare_cd_o3_mini_predictions.csv


 18%|█████████████▉                                                                 | 40/226 [28:52<3:42:35, 71.80s/it]

[checkpoint] saved 40/226 -> /content/flare_cd_o3_mini_predictions.csv


...

[checkpoint] saved 200/226 -> /content/flare_cd_o3_mini_predictions.csv


 97%|███████████████████████████████████████████████████████████████████████████▉  | 220/226 [4:06:18<06:46, 67.71s/it]

[checkpoint] saved 220/226 -> /content/flare_cd_o3_mini_predictions.csv


100%|██████████████████████████████████████████████████████████████████████████████| 226/226 [4:13:04<00:00, 67.19s/it]

[done] o3-mini CD evaluation completed -> /content/flare_cd_o3_mini_predictions.csv
[errors] 204 errors logged -> /content/flare_cd_o3_mini_errors.csv


In [5]:
# Step 5: Compute evaluation metrics for token-level causal annotation
%pip install -q scikit-learn

import pandas as pd
import json
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# 加载预测结果
df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")

# 解析标签
def parse_labels(s):
    if pd.isna(s) or s == '[]' or s == '':
        return []
    try:
        return json.loads(s)
    except:
        return []

df['gold_parsed'] = df['gold_labels'].apply(parse_labels)
df['pred_parsed'] = df['pred_labels'].apply(parse_labels)

# 只考虑成功的预测
success_df = df[df['success'] == True].copy() if 'success' in df.columns else df.copy()
print(f"Total samples: {len(df)}")
print(f"Successful predictions: {len(success_df)}")
print(f"Failed predictions: {len(df) - len(success_df)}")

if len(success_df) > 0:
    # 收集所有token级别的标签
    all_true = []
    all_pred = []
    
    for _, row in success_df.iterrows():
        gold = row['gold_parsed']
        pred = row['pred_parsed']
        
        # 确保长度一致
        min_len = min(len(gold), len(pred))
        all_true.extend(gold[:min_len])
        all_pred.extend(pred[:min_len])
    
    # 定义标签类别
    labels = ["O", "B-CAUSE", "I-CAUSE", "B-EFFECT", "I-EFFECT"]
    
    # 计算指标
    accuracy = accuracy_score(all_true, all_pred)
    f1_macro = f1_score(all_true, all_pred, labels=labels, average='macro', zero_division=0)
    f1_weighted = f1_score(all_true, all_pred, labels=labels, average='weighted', zero_division=0)
    
    print("\n" + "="*50)
    print("EVALUATION RESULTS - o3-mini on FLARE-CD")
    print("="*50)
    print(f"Total Tokens: {len(all_true)}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Macro: {f1_macro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(all_true, all_pred, labels=labels, zero_division=0))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(all_true, all_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    print(cm_df)
    
    # 保存评估结果
    eval_results = {
        "model": MODEL,
        "dataset": "ChanceFocus/flare-cd",
        "split": "test",
        "task": "token-level causal annotation",
        "total_samples": len(df),
        "successful_predictions": len(success_df),
        "failed_predictions": len(df) - len(success_df),
        "total_tokens": len(all_true),
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "classification_report": classification_report(all_true, all_pred, labels=labels, zero_division=0, output_dict=True),
        "confusion_matrix": cm.tolist(),
        "labels": labels,
        "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
    }
    
    eval_path = f"{save_dir}/{run_tag}_evaluation_results.json"
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=2)
    print(f"\nEvaluation results saved -> {eval_path}")
    
else:
    print("No successful predictions to evaluate!")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Total samples: 226
Successful predictions: 22
Failed predictions: 204

EVALUATION RESULTS - o3-mini on FLARE-CD
Total Tokens: 891
Accuracy: 0.4411
F1-Macro: 0.3539
F1-Weighted: 0.5042

Classification Report:
              precision    recall  f1-score   support

           O       0.14      0.51      0.22       117
     B-CAUSE       0.22      0.20      0.21        20
     I-CAUSE       0.90      0.47      0.62       398
    B-EFFECT       0.21      0.24      0.22        17
    I-EFFECT       0.65      0.41      0.50       339

    accuracy                           0.44       891
   macro avg       0.43      0.37      0.35       891
weighted avg       0.68      0.44      0.50       891


Confusion Matrix:
            O  B-CAUSE  I-CAUSE  B-EFFECT  I-EFFECT
O          60        4        5         4        44
B-CAUSE    12        4        4         0         0
I-CAUSE   174        8      186         3        27
B-EFFECT  